# Gemma 4 LoRA Fine-Tune + HuggingFace Publish (Steps 29-30)

Fine-tune Gemma on your own domain data with LoRA (low-rank adaptation) — a
parameter-efficient method that trains small adapter weights rather than the full
model. Then publish the adapter to your `shootstuff` HuggingFace account and
optionally push an Ollama-compatible GGUF for local use.

## What you need
- GPU with ≥8GB VRAM (for `gemma4:2b`) or ≥16GB (for `gemma4:12b`)
- `transformers`, `peft`, `trl`, `bitsandbytes`, `datasets`, `accelerate`
- `huggingface_hub` with `HF_TOKEN` set
- `llama.cpp` (for GGUF conversion, optional)

```bash
pip install transformers peft trl bitsandbytes datasets accelerate huggingface_hub
export HF_TOKEN=hf_...
```

> **Note**: Fine-tuning requires a GPU. The cells in this notebook run end-to-end
> on a machine with a CUDA GPU. CPU-only machines can prepare datasets and push
> to HuggingFace but cannot run training.

## Part 1 — Dataset Preparation

In [ ]:
import json
import os
from pathlib import Path

HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_USERNAME = os.environ.get("HF_USERNAME", "shootstuff")
BASE_MODEL = "google/gemma-2-2b-it"  # use gemma-2-9b-it for the 12B equivalent

if not HF_TOKEN:
    print("Set HF_TOKEN to push to HuggingFace.")

In [ ]:
# Build a fine-tuning dataset in chat format
# Format: instruction-response pairs in Gemma chat template

EXAMPLE_DATASET = [
    {
        "instruction": "Analyse this Ethereum wallet and flag any unusual activity.",
        "input": "Address 0xABC received 50 ETH from a mixer, then split to 10 addresses within 1 hour.",
        "output": "This pattern is characteristic of mixer-obfuscated funds being distributed. Red flags: (1) funds originated from a known mixer, (2) rapid fan-out to multiple addresses within 60 minutes. Recommend: do not interact with this wallet. Monitor downstream addresses for further movement."
    },
    {
        "instruction": "Write a Shopify product description for a soy candle.",
        "input": "Name: Midnight Pine. Notes: cedarwood, black spruce, smoky vanilla. 8oz, 45-hour burn, hand-poured.",
        "output": "Breathe in the depth of a midnight forest with Midnight Pine — a hand-poured soy candle that blends cedarwood, black spruce, and a whisper of smoky vanilla. Each 8oz vessel delivers up to 45 hours of clean, even burn, filling your space with the quiet strength of old-growth woods. Perfect for winding down after a long day or creating an atmosphere that's effortlessly grounded."
    },
    {
        "instruction": "Extract line items from this receipt text.",
        "input": "TRADER JOE'S\n06/10/2026\nOrganic Bananas 0.89\nAlmond Milk 3.49\nSourdough Bread 4.99\nTax 0.27\nTOTAL 9.64",
        "output": '{"store": "TRADER JOE\'S", "date": "2026-06-10", "items": [{"name": "Organic Bananas", "amount": 0.89}, {"name": "Almond Milk", "amount": 3.49}, {"name": "Sourdough Bread", "amount": 4.99}], "tax": 0.27, "total": 9.64}'
    },
]

# Convert to Gemma chat template format
def format_for_gemma(example: dict) -> str:
    user_content = example["instruction"]
    if example.get("input"):
        user_content += f"\n\n{example['input']}"
    return (
        f"<start_of_turn>user\n{user_content}<end_of_turn>\n"
        f"<start_of_turn>model\n{example['output']}<end_of_turn>"
    )

formatted = [format_for_gemma(ex) for ex in EXAMPLE_DATASET]
print(f"Formatted {len(formatted)} training examples.")
print("\nExample:")
print(formatted[0][:300], "...")

In [ ]:
# Save dataset as JSONL
DATASET_PATH = "gemma4_finetune_dataset.jsonl"
with open(DATASET_PATH, "w") as f:
    for ex in EXAMPLE_DATASET:
        f.write(json.dumps(ex) + "\n")
print(f"Dataset saved to {DATASET_PATH} ({len(EXAMPLE_DATASET)} examples)")
print("\nFor a real fine-tune, you need 100-1000+ high-quality examples.")
print("Good sources for your use case:")
print("  - Your Shopify order/product history (export as CSV, convert to Q&A pairs)")
print("  - Your saved blockchain analysis notes")
print("  - Gmail exports of emails you've written (as 'draft this email' training)")

## Part 2 — LoRA Fine-Tuning (GPU Required)

In [ ]:
import sys

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    if GPU_AVAILABLE:
        print(f"GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
    else:
        print("No GPU detected. Training cells will be skipped.")
        print("Run this notebook on a machine with a CUDA GPU.")
except ImportError:
    GPU_AVAILABLE = False
    print("PyTorch not installed — install with: pip install torch")

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping training setup — GPU required.")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
    from peft import LoraConfig, get_peft_model, TaskType
    from trl import SFTTrainer
    from datasets import Dataset
    import torch

    # 4-bit quantised base model (fits in 8GB VRAM)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    print(f"Loading {BASE_MODEL} in 4-bit...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN,
    )
    print("Model loaded.")

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping LoRA config — GPU required.")
else:
    # LoRA configuration
    lora_config = LoraConfig(
        r=16,                    # rank — higher = more capacity but more VRAM
        lora_alpha=32,           # scaling factor
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # attention layers
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # Output: trainable params: ~4M of 2B total (~0.2%) — that's LoRA's efficiency

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping training — GPU required.")
else:
    # Prepare dataset
    train_data = Dataset.from_dict({"text": [format_for_gemma(ex) for ex in EXAMPLE_DATASET]})

    training_args = TrainingArguments(
        output_dir="./gemma4-lora-adapter",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch size = 8
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        report_to="none",  # disable wandb
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_data,
        args=training_args,
        tokenizer=tokenizer,
        dataset_text_field="text",
        max_seq_length=2048,
    )

    print("Starting training...")
    trainer.train()
    trainer.save_model("./gemma4-lora-adapter")
    print("Training complete. Adapter saved to ./gemma4-lora-adapter")

## Part 3 — Test the Fine-Tuned Adapter

In [ ]:
if not GPU_AVAILABLE:
    print("Skipping inference test — GPU required.")
else:
    from peft import PeftModel

    # Load base + adapter
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN
    )
    ft_model = PeftModel.from_pretrained(base, "./gemma4-lora-adapter")
    ft_model.eval()

    def generate(prompt: str, max_new_tokens: int = 300) -> str:
        formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
        inputs = tokenizer(formatted, return_tensors="pt").to(ft_model.device)
        with torch.no_grad():
            outputs = ft_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(generate("Write a product description for a lavender pillow spray."))

## Part 4 — Push Adapter to HuggingFace (Step 30)

In [ ]:
from huggingface_hub import HfApi, login

if HF_TOKEN:
    login(token=HF_TOKEN)
    api = HfApi()
    print(f"Logged in as: {api.whoami()['name']}")
else:
    print("Set HF_TOKEN to push to HuggingFace.")

In [ ]:
ADAPTER_REPO = f"{HF_USERNAME}/gemma4-2b-lora-personal"

if HF_TOKEN and Path("./gemma4-lora-adapter").exists():
    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=ADAPTER_REPO, repo_type="model", private=True, exist_ok=True)
        print(f"Repo ready: https://huggingface.co/{ADAPTER_REPO}")
    except Exception as e:
        print(f"Repo creation: {e}")

    # Push the adapter weights
    api.upload_folder(
        folder_path="./gemma4-lora-adapter",
        repo_id=ADAPTER_REPO,
        repo_type="model",
        commit_message="Add Gemma 4 2B LoRA adapter — personal domain fine-tune",
    )
    print(f"Adapter pushed to https://huggingface.co/{ADAPTER_REPO}")
elif not HF_TOKEN:
    print("Set HF_TOKEN to push.")
else:
    print("Run training first to generate the adapter at ./gemma4-lora-adapter")

In [ ]:
# Write a model card
MODEL_CARD = f"""---
base_model: {BASE_MODEL}
library_name: peft
tags:
  - lora
  - gemma
  - text-generation
  - personal
license: gemma
---

# Gemma 4 2B LoRA — Personal Domain Adapter

A LoRA fine-tune of [`{BASE_MODEL}`](https://huggingface.co/{BASE_MODEL}) on personal
domain data covering crypto/DeFi analysis, e-commerce copywriting, and document extraction.

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("{BASE_MODEL}")
model = PeftModel.from_pretrained(base, "{ADAPTER_REPO}")
tokenizer = AutoTokenizer.from_pretrained("{BASE_MODEL}")
```

## Training
- Method: LoRA (r=16, alpha=32)
- Target modules: q_proj, v_proj, k_proj, o_proj
- Quantisation: 4-bit NF4 (bitsandbytes)
- Epochs: 3
"""

Path("README_adapter.md").write_text(MODEL_CARD)

if HF_TOKEN:
    api.upload_file(
        path_or_fileobj="README_adapter.md",
        path_in_repo="README.md",
        repo_id=ADAPTER_REPO,
        repo_type="model",
    )
    print("Model card pushed.")
else:
    print("Model card written to README_adapter.md (set HF_TOKEN to push).")

## Part 5 — Convert to GGUF for Ollama (Optional)

In [ ]:
print("""To run your fine-tuned adapter locally in Ollama:

1. Merge LoRA adapter into base model weights:

   from peft import PeftModel
   merged = ft_model.merge_and_unload()
   merged.save_pretrained("./gemma4-merged")
   tokenizer.save_pretrained("./gemma4-merged")

2. Convert to GGUF with llama.cpp:

   git clone https://github.com/ggerganov/llama.cpp
   cd llama.cpp && make
   python convert_hf_to_gguf.py ../gemma4-merged --outfile gemma4-ft.gguf --outtype q4_k_m

3. Create Modelfile:

   FROM ./gemma4-ft.gguf
   PARAMETER num_ctx 32768

4. Load into Ollama:

   ollama create gemma4-personal -f Modelfile
   ollama run gemma4-personal

5. Use in LlamaIndex:

   from llama_index.llms.ollama import Ollama
   llm = Ollama(model="gemma4-personal", request_timeout=120.0)
""")

## Part 6 — Dataset Builder Helper

Utility to convert your existing data into fine-tuning pairs.

In [ ]:
import csv


def csv_to_finetune_pairs(
    csv_path: str,
    instruction_col: str,
    output_col: str,
    input_col: str = "",
    instruction_prefix: str = "",
) -> list[dict]:
    """Convert a CSV file to instruction-tuning pairs."""
    pairs = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            instruction = (instruction_prefix + " " + row[instruction_col]).strip()
            pairs.append({
                "instruction": instruction,
                "input": row.get(input_col, "") if input_col else "",
                "output": row[output_col],
            })
    return pairs


def gmail_threads_to_pairs(threads: list[dict]) -> list[dict]:
    """Convert email threads to 'draft a reply' fine-tuning pairs."""
    pairs = []
    for thread in threads:
        if len(thread.get("messages", [])) >= 2:
            original = thread["messages"][0].get("body", "")
            reply = thread["messages"][1].get("body", "")
            pairs.append({
                "instruction": "Draft a reply to this email.",
                "input": original[:500],
                "output": reply[:500],
            })
    return pairs


print("Dataset builder helpers ready.")
print("\nShopify product example:")
print("  csv_to_finetune_pairs('products.csv', 'Title', 'Body (HTML)',")
print("      instruction_prefix='Write a product description for:')")